In [14]:
from google.colab import drive
import pandas as pd
import numpy as np
import rpy2

In [15]:
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [16]:
asym_data = pd.read_csv('/content/drive/MyDrive/pioneer_data/asym_data.csv')
asym_data.head()

,dir_num,convo_id,left_id,left_enjoy,right_id,right_enjoy,enjoy_diff
0,3,46f8e9b8-f80a-48cf-90a0-2e29908202c0,5d5eeb06d8bcde00162d73f1,2.0,57656f6c2bfddf000125cce5,6.0,-4.0
1,4,a4fa5355-74ba-4622-a58b-57335efc8a9e,56802e5fc5767f00121cc6a0,9.0,5ecf4b18f7b0443609e07646,5.0,4.0
2,7,5a307dec-0265-4dab-ade6-bc6392695c9e,5e7072e5fb136c63e94f3fa4,5.0,5ca6bbf13b5fcf00100996e9,9.0,-4.0
3,8,7f717277-d9ae-4dbc-b520-0b741d91a6c7,5dae16d241fbb6001160ce72,3.0,5ea9c6541eb4f0121a911e1a,7.0,-4.0
4,10,d9e1f5a5-e4eb-43df-910b-d9ae2befb039,5f3864a4596925371d23631e,5.0,5bf3761862e1bc0001f15cb2,9.0,-4.0


In [17]:
sample = pd.read_csv(f'/content/drive/MyDrive/au_activity/{asym_data.loc[0]['convo_id']}/left.csv')
sample.head()

,frame,approx_time,FaceScore,input,AU01,AU02,AU04,AU05,AU06,AU07,...,AU14,AU15,AU17,AU20,AU23,AU24,AU25,AU26,AU28,AU43
0,0,00:00,0.0,data/temp/46f8e9b8-f80a-48cf-90a0-2e29908202c0...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5,00:00,0.0,data/temp/46f8e9b8-f80a-48cf-90a0-2e29908202c0...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10,00:00,0.0,data/temp/46f8e9b8-f80a-48cf-90a0-2e29908202c0...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,15,00:00,0.0,data/temp/46f8e9b8-f80a-48cf-90a0-2e29908202c0...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20,00:00,0.0,data/temp/46f8e9b8-f80a-48cf-90a0-2e29908202c0...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
THRESH = 0.95
MIN_FRAMES = 30
BASE = '/content/drive/MyDrive/au_activity'

skipped = [] # convos that fail the 30-frame check; manually review if necessary

for i, r in enumerate(asym_data.itertuples(), start=1):
  print(f'processing: {i}/{len(asym_data)}')
  left = f'{BASE}/{r.convo_id}/left.csv'
  right = f'{BASE}/{r.convo_id}/right.csv'

  left_df = pd.read_csv(left)
  right_df = pd.read_csv(right)

  # merge them into one df temporarily for easier access

  merged = pd.merge(left_df, right_df, on='frame', suffixes=('_L', '_R'))
  merged = merged.sort_values('frame').reset_index(drop=True)

  # find where both facescores >= 0.95 for at least 30 consecutive frames
  left_ok = merged['FaceScore_L'] >= THRESH
  right_ok = merged['FaceScore_R'] >= THRESH
  both_ok = left_ok & right_ok

  runs = (both_ok != both_ok.shift()).cumsum() # id each consecutive block
  run_len = both_ok.groupby(runs).transform('size') # get length of that block
  stable = both_ok & (run_len >= MIN_FRAMES)

  if not stable.any():
    skipped.append(r.convo_id)
    print(f'no stable frame windows: {r.convo_id}')
    continue

  start = stable.idxmax() # first frame of first stable run
  merged = merged.iloc[start:].reset_index(drop=True) # delete previous frames

  # unmerge the two dfs

  # separate by suffix
  left_cols = [c for c in merged.columns if c.endswith('_L')]
  right_cols = [c for c in merged.columns if c.endswith('_R')]

  # unmerge but make sure frame col is included
  p_left = merged[['frame'] + left_cols].copy()
  p_right = merged[['frame'] + right_cols].copy()

  # rename columns w/out suffix
  p_left.columns = ['frame'] + [c.removesuffix('_L') for c in left_cols]
  p_right.columns = ['frame'] + [c.removesuffix('_R') for c in right_cols]

  p_left.to_csv(f'{BASE}/{r.convo_id}/p_left.csv', index=False)
  p_right.to_csv(f'{BASE}/{r.convo_id}/p_right.csv', index=False)

print(f'processed {len(asym_data) - len(skipped)}/{len(asym_data)} convos')
print(f'{len(skipped)} conversations flagged; please review')

processing: 1/123
processing: 2/123
processing: 3/123
processing: 4/123
processing: 5/123
processing: 6/123
processing: 7/123
processing: 8/123
processing: 9/123
processing: 10/123
processing: 11/123
processing: 12/123
processing: 13/123
processing: 14/123
processing: 15/123
processing: 16/123
processing: 17/123
processing: 18/123
processing: 19/123
processing: 20/123
processing: 21/123
processing: 22/123
processing: 23/123
processing: 24/123
processing: 25/123
processing: 26/123
processing: 27/123
processing: 28/123
processing: 29/123
processing: 30/123
processing: 31/123
processing: 32/123
processing: 33/123
processing: 34/123
processing: 35/123
processing: 36/123
processing: 37/123
processing: 38/123
processing: 39/123
processing: 40/123
processing: 41/123
processing: 42/123
processing: 43/123
processing: 44/123
processing: 45/123
processing: 46/123
processing: 47/123
processing: 48/123
processing: 49/123
processing: 50/123
processing: 51/123
processing: 52/123
processing: 53/123
pr

In [ ]:
# conversation 113 (cid: 7fc59ebb-a980-4df1-9d39-f15ae25f3f27) was reviewed
# manually and the left speaker (uid: 5dd8cb39dba6348743ec0915) had their
# camera off for a majority of the video.

In [21]:
asym_data = asym_data[asym_data['convo_id'] != '7fc59ebb-a980-4df1-9d39-f15ae25f3f27']
asym_data.to_csv('/content/drive/MyDrive/pioneer_data/asym_data.csv', index=False)